In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [50]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error , mean_absolute_error

In [2]:
df = pd.read_parquet("drive/MyDrive/green_tripdata_2026-01.parquet")

In [3]:
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,1,2026-01-01 00:27:58,2026-01-01 00:55:16,N,1.0,65,233,2.0,6.20,31.7,...,1.5,7.5,0.0,NaN,1.0,45.20,1.0,1.0,2.75,0.75
1,2,2026-01-01 00:44:33,2026-01-01 01:32:56,N,5.0,66,188,5.0,5.36,50.0,...,0.0,10.2,0.0,NaN,1.0,61.20,1.0,2.0,0.00,0.00
2,1,2026-01-01 00:23:45,2026-01-01 00:45:03,N,1.0,65,179,4.0,10.60,41.5,...,1.5,2.0,0.0,NaN,1.0,46.00,1.0,1.0,0.00,0.00
3,1,2026-01-01 00:44:33,2026-01-01 01:00:45,N,1.0,42,141,1.0,4.20,19.8,...,1.5,0.0,0.0,NaN,1.0,25.05,2.0,1.0,2.75,0.00
4,2,2026-01-01 00:46:04,2026-01-01 01:04:40,N,1.0,95,82,1.0,2.76,19.1,...,0.5,0.0,0.0,NaN,1.0,21.60,2.0,1.0,0.00,0.00


In [4]:
print(df.shape,df.size)

(40272, 21) 845712


In [5]:
df.isna().sum()

,0
VendorID,0
lpep_pickup_datetime,0
lpep_dropoff_datetime,0
store_and_fwd_flag,5414
RatecodeID,5414
PULocationID,0
DOLocationID,0
passenger_count,5414
trip_distance,0
fare_amount,0


In [6]:
df['ehail_fee'].describe()

,ehail_fee
count,0.0
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN


In [7]:
df = df.drop(columns=['ehail_fee','store_and_fwd_flag','VendorID','payment_type'] , axis=1)

In [8]:
df.isna().sum()

,0
lpep_pickup_datetime,0
lpep_dropoff_datetime,0
RatecodeID,5414
PULocationID,0
DOLocationID,0
passenger_count,5414
trip_distance,0
fare_amount,0
extra,0
mta_tax,0


In [9]:
drop_cols = [
    "total_amount",
    "tip_amount",
    "tolls_amount",
    "fare_amount",
]

df = df.drop(columns=drop_cols)

In [10]:
df.isna().sum()


,0
lpep_pickup_datetime,0
lpep_dropoff_datetime,0
RatecodeID,5414
PULocationID,0
DOLocationID,0
passenger_count,5414
trip_distance,0
extra,0
mta_tax,0
improvement_surcharge,0


In [11]:
df["trip_duration"] = (
    df["lpep_dropoff_datetime"]
    - df["lpep_pickup_datetime"]
).dt.total_seconds()

In [12]:
df = df.drop(columns=["lpep_dropoff_datetime"])

In [13]:
df["pickup_hour"] = df["lpep_pickup_datetime"].dt.hour
df["pickup_day"] = df["lpep_pickup_datetime"].dt.dayofweek
df["pickup_month"] = df["lpep_pickup_datetime"].dt.month

In [14]:
df.isna().sum()

,0
lpep_pickup_datetime,0
RatecodeID,5414
PULocationID,0
DOLocationID,0
passenger_count,5414
trip_distance,0
extra,0
mta_tax,0
improvement_surcharge,0
trip_type,5415


## **Fill The Missing**

In [15]:
df["RatecodeID"] = df["RatecodeID"].fillna(-1)
df["passenger_count"] = df["passenger_count"].fillna(
    df["passenger_count"].median()
)
df["trip_type"] = df["trip_type"].fillna(-1)
df["trip_type"] = df["trip_type"].fillna("Unknown")
df["congestion_surcharge"] = df["congestion_surcharge"].fillna(0)

In [16]:
df.isna().sum()

,0
lpep_pickup_datetime,0
RatecodeID,0
PULocationID,0
DOLocationID,0
passenger_count,0
trip_distance,0
extra,0
mta_tax,0
improvement_surcharge,0
trip_type,0


In [17]:
df.head()

,lpep_pickup_datetime,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,extra,mta_tax,improvement_surcharge,trip_type,congestion_surcharge,cbd_congestion_fee,trip_duration,pickup_hour,pickup_day,pickup_month
0,2026-01-01 00:27:58,1.0,65,233,2.0,6.20,4.50,1.5,1.0,1.0,2.75,0.75,1638.0,0,3,1
1,2026-01-01 00:44:33,5.0,66,188,5.0,5.36,0.00,0.0,1.0,2.0,0.00,0.00,2903.0,0,3,1
2,2026-01-01 00:23:45,1.0,65,179,4.0,10.60,1.00,1.5,1.0,1.0,0.00,0.00,1278.0,0,3,1
3,2026-01-01 00:44:33,1.0,42,141,1.0,4.20,3.75,1.5,1.0,1.0,2.75,0.00,972.0,0,3,1
4,2026-01-01 00:46:04,1.0,95,82,1.0,2.76,1.00,0.5,1.0,1.0,0.00,0.00,1116.0,0,3,1


In [18]:
df.drop(columns= 'lpep_pickup_datetime',inplace=True)

In [19]:
df.head()

,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,extra,mta_tax,improvement_surcharge,trip_type,congestion_surcharge,cbd_congestion_fee,trip_duration,pickup_hour,pickup_day,pickup_month
0,1.0,65,233,2.0,6.20,4.50,1.5,1.0,1.0,2.75,0.75,1638.0,0,3,1
1,5.0,66,188,5.0,5.36,0.00,0.0,1.0,2.0,0.00,0.00,2903.0,0,3,1
2,1.0,65,179,4.0,10.60,1.00,1.5,1.0,1.0,0.00,0.00,1278.0,0,3,1
3,1.0,42,141,1.0,4.20,3.75,1.5,1.0,1.0,2.75,0.00,972.0,0,3,1
4,1.0,95,82,1.0,2.76,1.00,0.5,1.0,1.0,0.00,0.00,1116.0,0,3,1


In [20]:
df['PU_DO'] = df['PULocationID'].astype(str) + '_' + df['DOLocationID'].astype(str)

In [21]:
df.head()

,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,extra,mta_tax,improvement_surcharge,trip_type,congestion_surcharge,cbd_congestion_fee,trip_duration,pickup_hour,pickup_day,pickup_month,PU_DO
0,1.0,65,233,2.0,6.20,4.50,1.5,1.0,1.0,2.75,0.75,1638.0,0,3,1,65_233
1,5.0,66,188,5.0,5.36,0.00,0.0,1.0,2.0,0.00,0.00,2903.0,0,3,1,66_188
2,1.0,65,179,4.0,10.60,1.00,1.5,1.0,1.0,0.00,0.00,1278.0,0,3,1,65_179
3,1.0,42,141,1.0,4.20,3.75,1.5,1.0,1.0,2.75,0.00,972.0,0,3,1,42_141
4,1.0,95,82,1.0,2.76,1.00,0.5,1.0,1.0,0.00,0.00,1116.0,0,3,1,95_82


In [22]:
df.drop(columns=['PULocationID','DOLocationID'],inplace=True)

In [42]:
df = df[
    (df["trip_duration"] >= 60) &
    (df["trip_duration"] <= 7200)
]

In [43]:
df.size

544152

In [44]:
df.head()

,RatecodeID,passenger_count,trip_distance,extra,mta_tax,improvement_surcharge,trip_type,congestion_surcharge,cbd_congestion_fee,trip_duration,pickup_hour,pickup_day,pickup_month,PU_DO
0,1.0,2.0,6.20,4.50,1.5,1.0,1.0,2.75,0.75,1638.0,0,3,1,65_233
1,5.0,5.0,5.36,0.00,0.0,1.0,2.0,0.00,0.00,2903.0,0,3,1,66_188
2,1.0,4.0,10.60,1.00,1.5,1.0,1.0,0.00,0.00,1278.0,0,3,1,65_179
3,1.0,1.0,4.20,3.75,1.5,1.0,1.0,2.75,0.00,972.0,0,3,1,42_141
4,1.0,1.0,2.76,1.00,0.5,1.0,1.0,0.00,0.00,1116.0,0,3,1,95_82


In [45]:
X = df[["PU_DO", "trip_distance"]]
y = df["trip_duration"]

In [47]:
# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Convert to dictionaries
train_dicts = X_train.to_dict(orient="records")
test_dicts = X_test.to_dict(orient="records")

# DictVectorizer
dv = DictVectorizer()

X_train = dv.fit_transform(train_dicts)
X_test = dv.transform(test_dicts)

print(X_train.shape)
print(X_test.shape)

(31094, 4937)
(7774, 4937)


## **Modelling**

In [52]:
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [53]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print("MAE:", mae)


MAE: 342.2209137771177
